In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from category_encoders import TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import RandomizedSearchCV

In [2]:
df = pd.read_csv("cars.csv")
df.head()

,Make,Model,Year,Engine Fuel Type,Engine HP,Engine Cylinders,Transmission Type,Driven_Wheels,Number of Doors,Market Category,Vehicle Size,Vehicle Style,highway MPG,city mpg,Popularity,MSRP
0,BMW,1 Series M,2011,premium unleaded (required),335.0,6.0,MANUAL,rear wheel drive,2.0,"Factory Tuner,Luxury,High-Performance",Compact,Coupe,26,19,3916,46135
1,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Convertible,28,19,3916,40650
2,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,High-Performance",Compact,Coupe,28,20,3916,36350
3,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Coupe,28,18,3916,29450
4,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,Luxury,Compact,Convertible,28,18,3916,34500


In [3]:
df.drop("Model",inplace=True,axis=1)

In [4]:
df

,Make,Year,Engine Fuel Type,Engine HP,Engine Cylinders,Transmission Type,Driven_Wheels,Number of Doors,Market Category,Vehicle Size,Vehicle Style,highway MPG,city mpg,Popularity,MSRP
0,BMW,2011,premium unleaded (required),335.0,6.0,MANUAL,rear wheel drive,2.0,"Factory Tuner,Luxury,High-Performance",Compact,Coupe,26,19,3916,46135
1,BMW,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Convertible,28,19,3916,40650
2,BMW,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,High-Performance",Compact,Coupe,28,20,3916,36350
3,BMW,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Coupe,28,18,3916,29450
4,BMW,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,Luxury,Compact,Convertible,28,18,3916,34500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11909,Acura,2012,premium unleaded (required),300.0,6.0,AUTOMATIC,all wheel drive,4.0,"Crossover,Hatchback,Luxury",Midsize,4dr Hatchback,23,16,204,46120
11910,Acura,2012,premium unleaded (required),300.0,6.0,AUTOMATIC,all wheel drive,4.0,"Crossover,Hatchback,Luxury",Midsize,4dr Hatchback,23,16,204,56670
11911,Acura,2012,premium unleaded (required),300.0,6.0,AUTOMATIC,all wheel drive,4.0,"Crossover,Hatchback,Luxury",Midsize,4dr Hatchback,23,16,204,50620
11912,Acura,2013,premium unleaded (recommended),300.0,6.0,AUTOMATIC,all wheel drive,4.0,"Crossover,Hatchback,Luxury",Midsize,4dr Hatchback,23,16,204,50920


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11914 entries, 0 to 11913
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Make               11914 non-null  object 
 1   Year               11914 non-null  int64  
 2   Engine Fuel Type   11911 non-null  object 
 3   Engine HP          11845 non-null  float64
 4   Engine Cylinders   11884 non-null  float64
 5   Transmission Type  11914 non-null  object 
 6   Driven_Wheels      11914 non-null  object 
 7   Number of Doors    11908 non-null  float64
 8   Market Category    8172 non-null   object 
 9   Vehicle Size       11914 non-null  object 
 10  Vehicle Style      11914 non-null  object 
 11  highway MPG        11914 non-null  int64  
 12  city mpg           11914 non-null  int64  
 13  Popularity         11914 non-null  int64  
 14  MSRP               11914 non-null  int64  
dtypes: float64(3), int64(5), object(7)
memory usage: 1.4+ MB


In [46]:
[df['Number of Doors']==2.]

[0         True
 1         True
 2         True
 3         True
 4         True
          ...  
 11909    False
 11910    False
 11911    False
 11912    False
 11913    False
 Name: Number of Doors, Length: 11914, dtype: bool]

In [6]:
df.columns

Index(['Make', 'Year', 'Engine Fuel Type', 'Engine HP', 'Engine Cylinders',
       'Transmission Type', 'Driven_Wheels', 'Number of Doors',
       'Market Category', 'Vehicle Size', 'Vehicle Style', 'highway MPG',
       'city mpg', 'Popularity', 'MSRP'],
      dtype='object')

In [51]:
df[(df["Make"]=='Bugatti') &(df['Number of Doors']==2.)]


,Make,Year,Engine Fuel Type,Engine HP,Engine Cylinders,Transmission Type,Driven_Wheels,Number of Doors,Market Category,Vehicle Size,Vehicle Style,highway MPG,city mpg,Popularity,MSRP
11362,Bugatti,2008,premium unleaded (required),1001.0,16.0,AUTOMATED_MANUAL,all wheel drive,2.0,"Exotic,High-Performance",Compact,Coupe,14,8,820,2065902
11363,Bugatti,2008,premium unleaded (required),1001.0,16.0,AUTOMATED_MANUAL,all wheel drive,2.0,"Exotic,High-Performance",Compact,Coupe,14,8,820,1500000
11364,Bugatti,2009,premium unleaded (required),1001.0,16.0,AUTOMATED_MANUAL,all wheel drive,2.0,"Exotic,High-Performance",Compact,Coupe,14,8,820,1705769


In [40]:
df['highway MPG'].describe()

count    11914.000000
mean        26.637485
std          8.863001
min         12.000000
25%         22.000000
50%         26.000000
75%         30.000000
max        354.000000
Name: highway MPG, dtype: float64

In [37]:
df['Vehicle Style'].unique()

array(['Coupe', 'Convertible', 'Sedan', 'Wagon', '4dr Hatchback',
       '2dr Hatchback', '4dr SUV', 'Passenger Minivan', 'Cargo Minivan',
       'Crew Cab Pickup', 'Regular Cab Pickup', 'Extended Cab Pickup',
       '2dr SUV', 'Cargo Van', 'Convertible SUV', 'Passenger Van'],
      dtype=object)

In [9]:
y = df["MSRP"]
X = df.drop("MSRP", axis=1)

In [10]:
num_cols = [
    "Year",
    "Engine HP",
    "Engine Cylinders",
    "Number of Doors",
    "highway MPG",
    "city mpg",
    "Popularity"
]

low_card_cols = [
    
    "Engine Fuel Type",
    "Transmission Type",
    "Driven_Wheels",
    "Vehicle Size",
    "Vehicle Style"
]

high_card_cols = [
    "Make",
    "Market Category"
]

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [12]:
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

In [13]:
low_cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [14]:
high_cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("target", TargetEncoder())
])

In [15]:
preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("low_cat", low_cat_pipe, low_card_cols),
    ("high_cat", high_cat_pipe, high_card_cols)
])

In [16]:
model_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestRegressor(
          n_estimators=400,
    max_depth=18,
    min_samples_split=8,
    min_samples_leaf=3,
    max_features="sqrt",
    bootstrap=True,
    random_state=42,
    n_jobs=-1
    ))
])

In [17]:
param_dist = {
    "model__n_estimators": [200, 300, 400, 500],
    
    "model__max_depth": [None, 10, 15, 20, 25],
    
    "model__min_samples_split": [2, 5, 10, 15],
    
    "model__min_samples_leaf": [1, 2, 4, 6],
    
    "model__max_features": ["sqrt", "log2", 0.5, 0.7],
    
    "model__bootstrap": [True]
}

In [18]:
random_search = RandomizedSearchCV(
    estimator=model_pipeline,
    param_distributions=param_dist,
    n_iter=30,          # good balance speed vs quality
    cv=5,
    scoring="r2",
    n_jobs=-1,
    random_state=42,
    verbose=2
)

In [19]:
# model_pipeline = Pipeline([
#     ("preprocess", preprocessor),
#     ("model", RandomForestRegressor(
#           n_estimators=400,
#     max_depth=18,
#     min_samples_split=8,
#     min_samples_leaf=3,
#     max_features="sqrt",
#     bootstrap=True,
#     random_state=42,
#     n_jobs=-1
#     ))
# ])

In [20]:
random_search.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


[CV] END model__bootstrap=True, model__max_depth=25, model__max_features=0.7, model__min_samples_leaf=2, model__min_samples_split=10, model__n_estimators=200; total time=   4.5s
[CV] END model__bootstrap=True, model__max_depth=15, model__max_features=0.5, model__min_samples_leaf=6, model__min_samples_split=5, model__n_estimators=500; total time=   9.1s
[CV] END model__bootstrap=True, model__max_depth=25, model__max_features=0.7, model__min_samples_leaf=2, model__min_samples_split=10, model__n_estimators=200; total time=  16.7s
[CV] END model__bootstrap=True, model__max_depth=25, model__max_features=0.7, model__min_samples_leaf=2, model__min_samples_split=10, model__n_estimators=200; total time=  17.1s
[CV] END model__bootstrap=True, model__max_depth=25, model__max_features=0.7, model__min_samples_leaf=2, model__min_samples_split=10, model__n_estimators=200; total time=  17.1s
[CV] END model__bootstrap=True, model__max_depth=25, model__max_features=0.7, model__min_samples_leaf=2, model_

,estimator,Pipeline(step...m_state=42))])
,param_distributions,"{'model__bootstrap': [True], 'model__max_depth': [None, 10, ...], 'model__max_features': ['sqrt', 'log2', ...], 'model__min_samples_leaf': [1, 2, ...], ...}"
,n_iter,30
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [21]:
# model_pipeline.fit(X_train, y_train)

In [22]:
pred = random_search.predict(X_test)

In [23]:
print("R2 Score:", r2_score(y_test, pred))
print("MAE:", mean_absolute_error(y_test, pred))

R2 Score: 0.9854587720808802
MAE: 2811.0595541886732


In [24]:
import joblib

joblib.dump(model_pipeline, "car_price_pipeline.pkl")

['car_price_pipeline.pkl']

In [25]:
# from sklearn.metrics import r2_score
# train_pred = random_search.predict(X_train)

# from sklearn.metrics import r2_score
# print("Train R2:", r2_score(y_train, train_pred))
# print("Test  R2:", round(r2_score(y_test, pred),5))

In [26]:
df

,Make,Year,Engine Fuel Type,Engine HP,Engine Cylinders,Transmission Type,Driven_Wheels,Number of Doors,Market Category,Vehicle Size,Vehicle Style,highway MPG,city mpg,Popularity,MSRP
0,BMW,2011,premium unleaded (required),335.0,6.0,MANUAL,rear wheel drive,2.0,"Factory Tuner,Luxury,High-Performance",Compact,Coupe,26,19,3916,46135
1,BMW,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Convertible,28,19,3916,40650
2,BMW,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,High-Performance",Compact,Coupe,28,20,3916,36350
3,BMW,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Coupe,28,18,3916,29450
4,BMW,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,Luxury,Compact,Convertible,28,18,3916,34500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11909,Acura,2012,premium unleaded (required),300.0,6.0,AUTOMATIC,all wheel drive,4.0,"Crossover,Hatchback,Luxury",Midsize,4dr Hatchback,23,16,204,46120
11910,Acura,2012,premium unleaded (required),300.0,6.0,AUTOMATIC,all wheel drive,4.0,"Crossover,Hatchback,Luxury",Midsize,4dr Hatchback,23,16,204,56670
11911,Acura,2012,premium unleaded (required),300.0,6.0,AUTOMATIC,all wheel drive,4.0,"Crossover,Hatchback,Luxury",Midsize,4dr Hatchback,23,16,204,50620
11912,Acura,2013,premium unleaded (recommended),300.0,6.0,AUTOMATIC,all wheel drive,4.0,"Crossover,Hatchback,Luxury",Midsize,4dr Hatchback,23,16,204,50920


In [27]:
import pickle



with open("car_price_pipeline.pkl", "wb") as f:
    pickle.dump(random_search, f)